In [78]:
# Checking Python executable
import sys
print(sys.executable) # MAKE SURE THIS POINTS TO THE CORRECT VIRTUAL ENVIRONMENT PATH FOR CORRECT PACKAGE INSTALLATION

/home/louis/miniconda3/envs/aml_lab/bin/python


In [79]:
# ALWAYS INSTALL USING %pip, NOT !pip (can sometimes install to system Python) or pip
# %pip install numpy
# %pip install pandas
# %pip install matplotlib
# %pip install scikit-learn
# %pip install torch # Using version 2.10.0+cu128
# %pip install torchinfo

In [80]:
# Import packages
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torchinfo
from preprocessing import create_training_dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import torch.nn.functional as F
print(torch.__version__)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu') # Define device (either GPU or CPU if GPU is unavailable)
# DEVICE = "cpu"

##### Model training function #####
def train(
        model: nn.Module,
        train_loader: DataLoader,
        criterion: nn.Module,
        optimizer: torch.optim.Optimizer,
        num_epochs: int = 10,
        val_loader: DataLoader = None,
        device: torch.device = DEVICE,
        print_loss: bool = True, # Flag for whether to print loss outputs or not
):
    model = model.to(device) # Move the model to same device as data (GPU or CPU)

    least_val_loss = 100 # Initialise best validation accuracy
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

    # Train for the number of epochs specified
    for epoch in range(num_epochs):

        ### TRAINING SET ###
        model.train() # Set model to training mode (affects Dropout/BatchNorm)
        train_loss = 0.0 # Initialise training loss

        # Loop through all batches in the training set
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device) # Move data to same device as model (GPU or CPU)
            optimizer.zero_grad() # Clear gradients
            preds = model(inputs) # Forward pass, obtain predictions
            loss = criterion(preds, labels) # Compute loss based on predictions and true labels (loss is MEAN loss over the batch)
            loss.backward() # Backward pass, compute gradient of loss w.r.t every model parameter
            optimizer.step() # Update weights, using optimisation algorithm chosen

            train_loss += loss.item() * inputs.size(0) # Sum training loss of EACH SAMPLE in the batch (inputs.size(0) is batch size)
        train_loss /= len(train_loader.dataset) # Calculate mean loss PER SAMPLE over ENTIRE DATASET

        ### VALIDATION SET ###
        # Loop through all validation batches (if validation data is given)
        if val_loader is not None:
            model.eval() # Set model to evaluation (inference) mode (turns dropout OFF, and affects BatchNorm)
            val_loss = 0.0 # Initialise validation loss
            all_preds_class = [] # Initialise list to store output prediction classes
            all_labels = [] # Initialise list to store actual labels of output predictions

            with torch.no_grad(): # Disable gradient computing
                # Loop through all batches in the validation set
                for inputs, labels in val_loader:
                    inputs, labels = inputs.to(device), labels.to(device) # Move data to same device as model (GPU or CPU)
                    preds = model(inputs) # Forward pass, obtain predictions as LOGITS (NO FOLLOWING BACKWARD PASS IN VALIDATION)

                    # Compute confusion matrix values
                    preds_class = torch.argmax(preds, dim=1) # Get class index of logit predictions
                    all_preds_class.append(preds_class.cpu())
                    all_labels.append(labels.cpu())

                    # Compute validation loss
                    loss = criterion(preds, labels) # Compute loss based on predictions and true labels (loss is MEAN loss over the batch)
                    val_loss += loss.item() * inputs.size(0) # Sum validation loss of EACH SAMPLE in the batch (inputs.size(0) is batch size)
                val_loss /= len(val_loader.dataset)

                all_preds_class = torch.cat(all_preds_class)
                all_labels = torch.cat(all_labels)

                cm = confusion_matrix(all_labels, all_preds_class)
                print("Validation confusion matrix:\n", cm)

                scheduler.step(val_loss)

                if val_loss < least_val_loss:
                    print("FOUND BEST")
                    least_val_loss = val_loss
                    torch.save(model.state_dict(), "Ml-Models/best_emotion_lstm_2.pth")

        ### PRINT TRAINING/VALIDATION OUTPUTS ###
            if print_loss:
                print(f"Epoch[{epoch+1}/{num_epochs}] Training Loss: {train_loss:.5f}, Validation Loss: {val_loss:.5f}")
        else:
            if print_loss:
                print(f"Epoch[{epoch+1}/{num_epochs}] Training Loss: {train_loss:.5f}")


##### Model evaluation function #####
def eval(
        model: nn.Module,
        test_loader: DataLoader,
        criterion: nn.Module,
        device: torch.device = DEVICE,
        print_loss: bool = True, # Flag for whether to print loss outputs or not
):
    model.eval() # Set model to evaluation mode
    test_loss = 0.0 # Initialise test loss
    all_preds_class = [] # Initialise list to store output prediction classes
    all_labels = [] # Initialise list to store actual labels of output predictions

    with torch.no_grad(): # Disable gradient computing
        # Loop through all batches in the test set
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device) # Move data to same device as model (GPU or CPU)
            preds = model(inputs) # Forward pass, obtain predictions as LOGITS (NO FOLLOWING BACKWARD PASS IN TESTING)

            # Compute confusion matrix values
            preds_class = torch.argmax(preds, dim=1) # Get class index of logit predictions
            all_preds_class.append(preds_class.cpu())
            all_labels.append(labels.cpu())
            
            # Compute validation loss
            loss = criterion(preds, labels) # Compute loss based on predictions and true labels (loss is MEAN loss over the batch)

            test_loss += loss.item() * inputs.size(0) # Sum test loss of EACH SAMPLE in the batch (inputs.size(0) is batch size)
        test_loss /= len(test_loader.dataset)

        all_preds_class = torch.cat(all_preds_class)
        all_labels = torch.cat(all_labels)

        cm = confusion_matrix(all_labels, all_preds_class)
        print("Testing confusion matrix:\n", cm)

    if print_loss:
        print(f"Test Loss: {test_loss:.5f}")

2.10.0+cu128


In [81]:
### Import data ###
data_dir = "EmoRecData/"

# Input channels, labels, one hot labels
X, y_int, label_reg=create_training_dataset(data_dir,None)

print(X.shape)

# # Remove first two columns (they are just host_time and time)
# # data1 = pd.read_csv("Data/adi_focused.csv")
# # data2 = pd.read_csv("Data/adi_stressed.csv")
# # data3 = pd.read_csv("Data/louis_focused.csv")
# # data4 = pd.read_csv("Data/louis_stressed.csv")
# base_data1_raw = pd.read_csv("Data/adi_7_5min_baseline.csv").iloc[:, 2:]
# base_data2_raw = pd.read_csv("Data/emmanuel_7_5min_baseline.csv").iloc[:, 2:]
# base_data3_raw = pd.read_csv("Data/louis_7_5min_baseline.csv").iloc[:, 2:]

# dist_data1_raw = pd.read_csv("Data/adi_7_5min_distraction.csv").iloc[:, 2:]
# dist_data2_raw = pd.read_csv("Data/emmanuel_7_5min_diistraction.csv").iloc[:, 2:]
# dist_data3_raw = pd.read_csv("Data/louis_7_5min_distract.csv").iloc[:, 2:]

# foc_data1_raw = pd.read_csv("Data/adi_7_5min_focus.csv").iloc[:, 2:]
# foc_data2_raw = pd.read_csv("Data/emmanuel_7_5min_focus.csv").iloc[:, 2:]
# foc_data3_raw = pd.read_csv("Data/louis_7_5min_focus.csv").iloc[:, 2:]

# str_data1_raw = pd.read_csv("Data/adi_7_5min_stress.csv").iloc[:, 2:]
# str_data2_raw = pd.read_csv("Data/emmanuel_7_5min_stress.csv").iloc[:, 2:]
# str_data3_raw = pd.read_csv("Data/louis_7_5min_stress.csv").iloc[:, 2:]

# # DEBUGGING
# # base_data1.head()

Skipping test file: adi_7_5min_baseline.csv
Skipping test file: adi_7_5min_distract.csv
Skipping test file: adi_7_5min_focus.csv
Skipping test file: adi_7_5min_stress.csv
Skipping test file: adi_focused.csv
Skipping test file: emmanuel_7_5min_baseline.csv
Skipping test file: emmanuel_7_5min_distract.csv
Skipping test file: emmanuel_7_5min_focus.csv
Skipping test file: emmanuel_7_5min_stress.csv
Skipping test file: focus_test.csv
Skipping test file: louis_7_5min_baseline.csv
Skipping test file: louis_7_5min_distract.csv
Skipping test file: louis_7_5min_focus.csv
Skipping test file: louis_7_5min_stress.csv
Skipping test file: louis_focused.csv
Skipping test file: louis_stressed.csv
Using 36 training files (ignored 16 test/backup files)
Processing adi1_7_5min_baseline.csv...
Processing adi1_7_5min_distract.csv...
Processing adi1_7_5min_focus.csv...
Processing adi1_7_5min_stress.csv...
Processing adi2_7_5min_baseline.csv...
Processing adi2_7_5min_distract.csv...
Processing adi2_7_5min_focu

In [82]:
label_encoder = LabelEncoder()

# Train val split
X_train, X_val, y_train, y_val = train_test_split(
    X, y_int, test_size=0.2, random_state=42, stratify=y_int
)

print(X_train.shape)

def infer_label_from_filename(fname: str) -> str:
    lower = fname.lower()
    for key, lab in label_map.items():
        if key in lower:
            return lab
    raise ValueError(f"Could not infer label for: {fname}")


(12960, 125, 14)


In [83]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

X_train_torch = torch.FloatTensor(X_train).transpose(1, 2) 
X_val_torch   = torch.FloatTensor(X_val).transpose(1, 2)
y_train_torch = torch.LongTensor(y_train)
y_val_torch   = torch.LongTensor(y_val)

print(f"X_train_torch shape: {X_train_torch.shape}")
print(f"X_val_torch shape:   {X_val_torch.shape}")

BATCH_SIZE = 64
train_dataset = TensorDataset(X_train_torch, y_train_torch)
val_dataset   = TensorDataset(X_val_torch,   y_val_torch)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

for batch_X, batch_y in train_loader:
    print(f"FIRST BATCH shape: {batch_X.shape}")
    break



# Initialisations
num_sensor_readings = 14 # Number of sensor readings
num_classes = 4 # Number of classification classes (number of emotional states to identify)

Using device: cuda
X_train_torch shape: torch.Size([12960, 14, 125])
X_val_torch shape:   torch.Size([3241, 14, 125])
FIRST BATCH shape: torch.Size([64, 14, 125])


In [84]:
# ##### Process data #####
# ### Function for normalising data ###
# def normalise_data(data_in): # INPUT DATA IS A PANDAS DATAFRAME
#     return (data_in - data_in.mean()) / data_in.std()

# ### Function for obtaining training, validation and test splits
# def split_data(data_in, train_split, val_split, test_split): # data_in is a list of pandas dataframes
#     # Create empty pandas dataframes for storing training, validation and test data splits
#     train_data = pd.DataFrame()
#     val_data = pd.DataFrame()
#     test_data = pd.DataFrame()

#     # Loop through all datasets in the input list of datasets
#     for dataset in data_in:
#         # Obtain number of training, validation and test data
#         train_num = round(dataset.shape[0]*train_split)
#         val_num = round(dataset.shape[0]*val_split)
#         test_num = dataset.shape[0] - train_num - val_num

#         # Obtain the training, validation and test data splits
#         train_data_curr = dataset.iloc[0:train_num]
#         val_data_curr = dataset.iloc[train_num:train_num+val_num]
#         test_data_curr = dataset.iloc[train_num+val_num:]

#         # Concatenate current dataset's training, validation and test data to the overall data
#         train_data = pd.concat([train_data, train_data_curr], axis=0, ignore_index=True)
#         val_data = pd.concat([val_data, val_data_curr], axis=0, ignore_index=True)
#         test_data = pd.concat([test_data, test_data_curr], axis=0, ignore_index=True)
    
#     # Convert pandas dataframes to numpy arrays
#     train_data_np = train_data.values.astype("float32")
#     val_data_np = val_data.values.astype("float32")
#     test_data_np = test_data.values.astype("float32")

#     return train_data_np, val_data_np, test_data_np

# ### Function for obtaining array of class labels for a dataset
# def create_labels(data_in, label):
#     all_labels = [] # Initialise

#     # Loop through all training, validation and test datasets
#     for dataset in data_in:
#         dataset_labels = np.ones(len(dataset), dtype=np.int64)*label # Create an array of desired labels for the current dataset
#         all_labels.append(dataset_labels) # Append labels to output

#     return all_labels[0], all_labels[1], all_labels[2] # Return array of labels for training, validation and test sets

# ### Function for obtaining windowed input-label pairs FOR A SINGLE DATASET (needed as our data is highly dependent on previous data) ###
# # E.g. [x0, x1, x2] -> y       [x1, x2, x3] -> y ...
# def window_single_data(dataset, labels, win_size=10):
#     # print(range(len(dataset) - win_size))
#     # print(dataset)
#     input_seq = np.array([dataset[i:i+win_size, :] for i in range(len(dataset) - win_size)]) # Get input sequence with length = window length
#     seq_label = [labels[i+win_size, 0] for i in range(len(dataset) - win_size)] # Get corresponding output for each input window sequence

#     # print(input_seq)
#     return np.array(input_seq), np.array(seq_label)

# ### Function for obtaining windowed input-label pairs FOR A LIST OF DATASETS ###
# def window_data(data_in, labels_in, win_size=10):
#     # Initialisations
#     all_data = []
#     all_labels = []

#     # Loop through all datasets (in the data_in list)
#     for dataset, labels in zip(data_in, labels_in):
#         windowed_dataset, windowed_labels = window_single_data(dataset, np.vstack(labels), win_size) # Get windowed input-label pairs for current dataset
#         all_data.append(windowed_dataset) # Store current windowed dataset
#         all_labels.append(windowed_labels) # Store current windowed labels
    
#     data_out = np.concatenate(all_data, axis=0) # Concatenate all windowed dataset
#     labels_out = np.concatenate(all_labels, axis=0) # Concatenate all windowed labels

#     return data_out, labels_out


# # Initialisations
# num_sensor_readings = 20 # Number of sensor readings
# num_classes = 4 # Number of classification classes (number of emotional states to identify)
# window_size = 16

# # Normalise all data
# base_data1 = normalise_data(base_data1_raw)
# base_data2 = normalise_data(base_data2_raw)
# base_data3 = normalise_data(base_data3_raw)

# dist_data1 = normalise_data(dist_data1_raw)
# dist_data2 = normalise_data(dist_data2_raw)
# dist_data3 = normalise_data(dist_data3_raw)

# foc_data1 = normalise_data(foc_data1_raw)
# foc_data2 = normalise_data(foc_data2_raw)
# foc_data3 = normalise_data(foc_data3_raw)

# str_data1 = normalise_data(str_data1_raw)
# str_data2 = normalise_data(str_data2_raw)
# str_data3 = normalise_data(str_data3_raw)

# # Group all data
# base_data = [base_data1, base_data2, base_data3]
# dist_data = [dist_data1, dist_data2, dist_data3]
# foc_data = [foc_data1, foc_data2, foc_data3]
# str_data = [str_data1, str_data2, str_data3]

# # Obtain training, validation and test splits
# base_train_data, base_val_data, base_test_data = split_data(base_data, 0.8, 0.1, 0.1)
# dist_train_data, dist_val_data, dist_test_data = split_data(dist_data, 0.8, 0.1, 0.1)
# foc_train_data, foc_val_data, foc_test_data = split_data(foc_data, 0.8, 0.1, 0.1)
# str_train_data, str_val_data, str_test_data = split_data(str_data, 0.8, 0.1, 0.1)
# # print(base_train_data)

# # Obtain labels for training, validation and test splits
# base_train_labels, base_val_labels, base_test_labels = create_labels([base_train_data, base_val_data, base_test_data], 0)
# dist_train_labels, dist_val_labels, dist_test_labels = create_labels([dist_train_data, dist_val_data, dist_test_data], 1)
# foc_train_labels, foc_val_labels, foc_test_labels = create_labels([foc_train_data, foc_val_data, foc_test_data], 2)
# str_train_labels, str_val_labels, str_test_labels = create_labels([str_train_data, str_val_data, str_test_data], 3)
# # print(str_test_labels)

# # Place all training, validation and testing data and labels into lists
# train_data_list = [base_train_data, dist_train_data, foc_train_data, str_train_data]
# train_labels_list = [base_train_labels, dist_train_labels, foc_train_labels, str_train_labels]

# val_data_list = [base_val_data, dist_val_data, foc_val_data, str_val_data]
# val_labels_list = [base_val_labels, dist_val_labels, foc_val_labels, str_val_labels]

# test_data_list = [base_test_data, dist_test_data, foc_test_data, str_test_data]
# test_labels_list = [base_test_labels, dist_test_labels, foc_test_labels, str_test_labels]

# # Window the training, validation and test splits
# train_data, train_labels = window_data(train_data_list, train_labels_list, window_size)
# val_data, val_labels = window_data(val_data_list, val_labels_list, window_size)
# test_data, test_labels = window_data(test_data_list, test_labels_list, window_size)
# print(train_data.shape)


# # ### DEBUGGING
# # # print(train_data_raw.head()) # Print first 5 rows of data for inspection
# # # print(test_data_raw.head()) # Print first 5 rows of data for inspection
# # # print(test_data.shape)
# # # plt.plot(train_data_np[:,2])

In [85]:
# ##### Create dataloaders #####
# # Convert data to tensors
# train_inputs_tensor = torch.tensor(train_data, dtype=torch.float32)
# train_labels_tensor = torch.tensor(train_labels, dtype=torch.long)#.unsqueeze(1) # nn.CrossEntropyLoss() expects integer class labels, no floats or one-hot. Also no need for unsqueeze(1) for CrossEntropyLoss, it just take labels with dim [batch size]

# val_inputs_tensor = torch.tensor(val_data, dtype=torch.float32)
# val_labels_tensor = torch.tensor(val_labels, dtype=torch.long)#.unsqueeze(1)

# test_inputs_tensor = torch.tensor(test_data, dtype=torch.float32)
# test_labels_tensor = torch.tensor(test_labels, dtype=torch.long)#.unsqueeze(1)

# # Build data loaders
# train_dataset = TensorDataset(train_inputs_tensor, train_labels_tensor)
# train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# val_dataset = TensorDataset(val_inputs_tensor, val_labels_tensor)
# val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)#True)

# test_dataset = TensorDataset(test_inputs_tensor, test_labels_tensor)
# test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)#True)


# ## DEBUGGING
# print(f"Shape of training inputs: {train_inputs_tensor.shape}")
# print(f"Shape of training labels: {train_labels_tensor.shape}")

# print(f"Shape of validation inputs: {val_inputs_tensor.shape}")
# print(f"Shape of validation labels: {val_labels_tensor.shape}")

# print(f"Shape of testing inputs: {test_inputs_tensor.shape}")
# print(f"Shape of testing labels: {test_labels_tensor.shape}")

In [86]:
# print(f"Train labels min: {train_labels_tensor.min()}, max: {train_labels_tensor.max()}")
# print(f"Train labels shape: {train_labels_tensor.shape}")

In [87]:
##### Model definition #####
class LSTMClassifier(nn.Module):
    def __init__(self, input_size=num_sensor_readings, hidden_size=64, output_size=num_classes): # Note: output_size should be equal to number of classification classes
        super().__init__()
        self.lstm1 = nn.LSTM(input_size=input_size, hidden_size=hidden_size, batch_first=True, dropout=0.3) # Input: [batch, sequence length, input dimension (number of sensors)]
        # self.relu1 = nn.ReLU() # ReLU
        self.dropout1 = nn.Dropout(0.2) # Dropout

        self.lstm2 = nn.LSTM(input_size=hidden_size, hidden_size=hidden_size, batch_first=True, dropout=0.3) # Input: [batch, sequence length, input dimension (number of sensors)]
        # self.relu2 = nn.ReLU() # ReLU
        self.dropout2 = nn.Dropout(0.2) # Dropout

        self.fc1 = nn.Linear(hidden_size, hidden_size)
        self.layernorm1 = nn.LayerNorm(hidden_size) # Layer norm
        self.relu3 = nn.ReLU() # ReLU
        self.dropout3 = nn.Dropout(0.2) # Dropout

        self.fc2 = nn.Linear(hidden_size, output_size)
        # self.layernorm2 = nn.LayerNorm(output_size) # Layer norm
        # self.relu4 = nn.ReLU() # ReLU
        # self.dropout4 = nn.Dropout(0.3) # Dropout
    
    def forward(self, x):
        x = x.permute(0,2,1)

        out, _ = self.lstm1(x) # Output: [batch, sequence length, hidden dimension (number of sensors)]
        # out = out[:, -1, :] # Use final hidden state of model as the output classification (Output: [batch, hidden dimension])
        # out = self.relu1(out)
        out = self.dropout1(out)
        
        out, _ = self.lstm2(out)
        # out = self.relu2(out)
        out = self.dropout2(out)
        out = out[:, -1, :]
        
        out = self.fc1(out) # CLASSIFY: Output here are LOGITS (Output: [batch, num_classes])
        out = self.layernorm1(out)
        out = self.relu3(out)
        out = self.dropout3(out)

        out = self.fc2(out) # CLASSIFY: Output here are LOGITS (Output: [batch, num_classes])
        # out = self.layernorm2(out)
        # out = self.relu4(out)
        # out_logits = self.dropout4(out)

        return out # AS LOGITS

In [88]:
##### Traing model #####
model = LSTMClassifier()#.to(DEVICE) # Define model
# print(torchinfo.summary(model, input_size=(1, window_size, num_sensor_readings))) # Input: [batch size, sequence length, input size (number of sensors)]

# Train model
train(
    model,
    train_loader,
    nn.CrossEntropyLoss(), #nn.CrossEntropyLoss() for multi-class classification, nn.BCEWithLogitsLoss() for binary classification, nn.MSELoss()
    optim.Adam(model.parameters(), lr=0.001),
    num_epochs=150,#500
    val_loader=val_loader,
    print_loss=True,
)


# # Test model
# eval(
#     model,
#     test_loader,
#     nn.CrossEntropyLoss(), #nn.CrossEntropyLoss() for multi-class classification, nn.BCEWithLogitsLoss() for binary classification, nn.MSELoss()
#     print_loss=True,
# )

/home/louis/miniconda3/envs/aml_lab/lib/python3.14/site-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.3 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


Validation confusion matrix:
 [[149 185  23 454]
 [ 32 654  43  78]
 [ 29 638  53  83]
 [106 314  18 382]]
FOUND BEST
Epoch[1/150] Training Loss: 1.36653, Validation Loss: 1.22744
Validation confusion matrix:
 [[214 588   4   5]
 [ 22 760  24   1]
 [ 15 738  49   1]
 [192 614   9   5]]
Epoch[2/150] Training Loss: 1.23784, Validation Loss: 1.32400
Validation confusion matrix:
 [[624   0 187   0]
 [320   5 481   1]
 [314   2 487   0]
 [567   2 250   1]]
Epoch[3/150] Training Loss: 1.22598, Validation Loss: 1.33406
Validation confusion matrix:
 [[609  29  70 103]
 [ 89 280 340  98]
 [158 189 355 101]
 [457 101 152 110]]
FOUND BEST
Epoch[4/150] Training Loss: 1.25698, Validation Loss: 1.20651
Validation confusion matrix:
 [[672   7 119  13]
 [117 162 519   9]
 [153  78 553  19]
 [546  19 243  12]]
FOUND BEST
Epoch[5/150] Training Loss: 1.19513, Validation Loss: 1.16129
Validation confusion matrix:
 [[322  46  84 359]
 [  4 506 208  89]
 [  5 358 341  99]
 [244 118 174 284]]
FOUND BEST
Epoc

In [104]:
def predict_file_reg(fpath: str, model_reg, label_encoder):
    """Predict emotions on new file using regularized model."""
    df = pd.read_csv(fpath)

    # Convert objects to numeric (same as training)
    for col in feature_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    data = df[feature_cols].dropna().values 

    windows = []
    start = 0
    while start + WINDOW_SIZE <= len(data):
        window = data[start:start + WINDOW_SIZE] 

        window_mean = np.mean(window, axis=0, keepdims=True)
        window_std  = np.std(window, axis=0, keepdims=True)
        window_norm = (window - window_mean) / (window_std + 1e-8)

        windows.append(window_norm)
        start += STEP_SIZE

    if not windows:
        return None, None

    X_new = np.stack(windows)  # (n_windows, 125, 14)
    X_new_torch = torch.FloatTensor(X_new).transpose(1, 2).to(device)  

    model_reg.eval()
    with torch.no_grad():
        outputs = model_reg(X_new_torch)
        probs = F.softmax(outputs, dim=1).cpu().numpy()
        preds = np.argmax(probs, axis=1)

    labels = label_encoder.inverse_transform(preds)
    return labels, probs

WINDOW_SIZE = 125
STEP_SIZE = 32


all_cols = ['host_time_s', 't_ms', 'ax1', 'ay1', 'az1', 'gx1', 'gy1', 'gz1', 'ax2', 'ay2', 'az2', 'gx2', 'gy2', 'gz2', 'emg1', 'emg2'] # without orientation data


time_cols = ["host_time_s", "t_ms"]
feature_cols = [c for c in all_cols if c not in time_cols]


# test_file = "/content/drive/MyDrive/EmoRecData/louis_stressed.csv" # change path
# test_file = "EmoRecData/louis_7_5min_focus.csv" # change path
test_file = "EmoRecData/louis_7_5min_baseline.csv" # change path
# test_file = "EmoRecData/louis_7_5min_stress.csv" # change path
# test_file = "EmoRecData/louis_7_5min_distract.csv" # change path

label_encoder = LabelEncoder()
# label_encoder.fit(["relaxed", "focused", "distracted", "stressed"])
label_encoder.fit(["distracted", "focused", "relaxed", "stressed"])
# labels_reg, probs_reg = predict_file_reg(test_file, model_reg, label_encoder)

# Call best model weights
model = model_reg
# model.load_state_dict(torch.load("Ml-Models/best_emotion_lstm_2.pth"))
model.load_state_dict(torch.load("Ml-Models/cnn_without_orientation_data.pth"))

labels_reg, probs_reg = predict_file_reg(test_file, model, label_encoder)

filename = os.path.basename(test_file)
print(f"🛡️ REGULARIZED MODEL (92% val acc) on {filename}:")
print(f"🎯 {len(labels_reg)} windows predicted")
print(f"Most common counts: {np.bincount([label_encoder.transform([l])[0] for l in labels_reg], minlength=4)}")

print("\nFirst 10 predictions:")
for i, pred in enumerate(labels_reg[:10]):
    print(f"  {i+1:2d}: {pred}")

print("\nPREDICTION DISTRIBUTION:")
unique, counts = np.unique(labels_reg, return_counts=True)
total_windows = len(labels_reg)
for label, count in zip(unique, counts):
    pct = count / total_windows * 100
    print(f"  {label:10s}: {count}/{total_windows} ({pct:.0f}%)")

# Top prediction
top_class_idx = np.argmax(np.bincount([label_encoder.transform([l])[0] for l in labels_reg]))
top_class = label_encoder.classes_[top_class_idx]
top_pct = 100 * np.max(np.bincount([label_encoder.transform([l])[0] for l in labels_reg])) / len(labels_reg)
true_label = infer_label_from_filename(filename)
print(f"\n TOP PREDICTION: {top_class} ({top_pct:.0f}%)")
print(f" TRUE LABEL:     {true_label}")

🛡️ REGULARIZED MODEL (92% val acc) on louis_7_5min_baseline.csv:
🎯 54 windows predicted
Most common counts: [ 0 26 24  4]

First 10 predictions:
   1: focused
   2: focused
   3: focused
   4: relaxed
   5: relaxed
   6: relaxed
   7: relaxed
   8: relaxed
   9: focused
  10: relaxed

PREDICTION DISTRIBUTION:
  focused   : 26/54 (48%)
  relaxed   : 24/54 (44%)
  stressed  : 4/54 (7%)


NameError: name 'label_map' is not defined

In [99]:
# Train REGULARIZED version
class EmotionCNN_Reg(nn.Module):
    def __init__(self, input_channels=14, num_classes=4):
        super().__init__()
        self.conv1 = nn.Conv1d(input_channels, 16, kernel_size=3, padding=1)
        self.bn1   = nn.BatchNorm1d(16)
        self.pool1 = nn.MaxPool1d(2)

        self.conv2 = nn.Conv1d(16, 32, kernel_size=3, padding=1)
        self.bn2   = nn.BatchNorm1d(32)
        self.pool2 = nn.MaxPool1d(2)

        #self.fc1       = nn.Linear(32 * 4, 64) # use for window size =16 / step size 4
        self.fc1       = nn.Linear(32 * 31, 64)
        self.dropout1  = nn.Dropout(0.3)
        self.fc2       = nn.Linear(64, 32)
        self.dropout2  = nn.Dropout(0.5)
        self.fc3       = nn.Linear(32, num_classes)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool1(x)
        x = F.dropout(x, 0.2)

        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool2(x)

        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout1(x)
        x = F.relu(self.fc2(x))
        x = self.dropout2(x)
        x = self.fc3(x)
        return x

# Retrain with regularization
model_reg = EmotionCNN_Reg(input_channels=X_train_torch.shape[1],
                           num_classes=4).to(device)
optimizer_reg = optim.Adam(model_reg.parameters(), lr=0.001, weight_decay=1e-3)
criterion = nn.CrossEntropyLoss()

print("🛡️ Training regularized model...")
print("Input channels:", X_train_torch.shape[1])
print("Total params:", sum(p.numel() for p in model_reg.parameters()))


🛡️ Training regularized model...
Input channels: 14
Total params: 68116
